# PPI example

One PPI run: data and split -> target -> inference -> output.

Run from `examples/simulation/`, so that `dataset` and `model` import directly.

In [1]:
from sklearn.ensemble import RandomForestRegressor

from openppi import split, PPI, Mean, PPIData
from dataset.judge_sim import make_judge_data

In [2]:
# 1. Split iid rows; fit only on the independent training sample.
frame = make_judge_data(n=2000, seed=0)
train, labeled, unlabeled = split(
    frame.drop(columns='label').to_numpy(), frame['label'].to_numpy(),
    train_size=400, labeled_size=400, random_state=42)
model = RandomForestRegressor(n_estimators=60, min_samples_leaf=8,
                              max_depth=8, random_state=0, n_jobs=1)
model.fit(*train)
data = PPIData(labels=labeled[1], labeled_predictions=model.predict(labeled[0]),
               unlabeled_predictions=model.predict(unlabeled))

In [3]:
# 2. Select the target.
target = Mean()

In [4]:
# 3. Combine two-sample residual correction with a Wald interval.
result = PPI()(data, target=target, alpha=.05)

In [5]:
# 4. Output.
print('Estimate:', result.estimate)
print('Confidence interval:', result.confidence_interval)
print('Diagnostics:', result.diagnostics)

Estimate: 0.4939071034182631
Confidence interval: (0.4456344198151464, 0.5421797870213798)
Diagnostics: {'labeled_count': 400, 'unlabeled_count': 1200, 'prediction_weight': 1.0, 'weight_strategy': 'fixed', 'variance_design': 'independent_two_sample'}
